# Stage 10 — Spark, Shuffle & Skew

**Goal:** להבין Partition Skew במחשב ללא צורך ב־Spark.

**Setup:** Python 3.11+, Jupyter/VS Code; local synthetic examples only.

**CS bridge:** Explain the invariant and its failure mode before running code.

**Source:** https://spark.apache.org/docs/latest/sql-performance-tuning.html

**Next:** Complete the graded lab under ../labs/ and pass an unseen transfer challenge.

## 1. Predict first

לפני Run All: כתוב מה אתה מצפה שה־assert יוכיח ומה אינו בודק. לאחר מכן הרץ את התא.

In [ ]:
from hashlib import sha256
def bucket(key,n):
    return int.from_bytes(sha256(key.encode()).digest()[:8],"big")%n
assert bucket("evt-A",4)==bucket("evt-A",4)
rows=[(f"evt-{i}","HOT" if i<90 else "COLD") for i in range(100)]
by_region={"HOT":0,"COLD":0}
for _,region in rows:
    by_region[region]+=1
assert by_region=={"HOT":90,"COLD":10}
print("Low-cardinality region skew:",by_region)

## 2. Failure mode / independent check

שנה פרמטר אחד בענף האישי והסבר מדוע הקוד נכשל או משנה תוצאה. החזר את הערך המקורי.

In [ ]:
partitions=[0]*4
for event_id,region in rows:
    partitions[bucket(event_id,4)]+=1
assert sum(partitions)==100
assert max(partitions)<90
print("High-cardinality event-id partition counts:",partitions)
print("Different partition key changes data movement; it does not guarantee better performance.")

## 3. Transfer (no AI)

אילו פעולות יוצרות Shuffle ב־Spark? כיצד מודדים Task Time ו־Spill במקום לנחש?

אל תסמן את השלב כהושלם על בסיס מחברת זו בלבד; חובה לבצע את המעבדה ובדיקת המנטור.